# UA5 - Aprendizaje Supervisado
## Predicción del Nivel de Obesidad

Este notebook es una **guía de trabajo** para desarrollar la actividad evaluativa de Aprendizaje Supervisado, usando el dataset `obesity_dataset.csv`.

**Variable objetivo:** `NObeyesdad` (nivel de obesidad) — tiene 7 categorías, por lo tanto este es un problema de **clasificación multiclase**.

Está basado en la lógica de los casos vistos en clase:
- **Caso 11** (regresión de arriendos): técnicas de EDA y preprocesamiento.
- **Caso 12** (clasificación de churn): estructura general para problemas de clasificación (referencia principal, porque también predice categorías).

> Recuerda: este notebook trae el código ya armado como guía. Complementa cada sección con tus propios comentarios, análisis e interpretaciones antes de entregarlo, para que refleje tu trabajo.

---
## 1. Librerías requeridas

Iniciamos importando las librerías que usaremos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

---
## 2. Importando los datos

Ajusta la ruta si tu archivo `obesity_dataset.csv` está en otra carpeta (por ejemplo `data/obesity_dataset.csv`, como se hizo en los casos vistos).

In [ ]:
df = pd.read_csv('obesity_dataset.csv')
df.head()

---
## 3. Análisis Exploratorio de Datos (EDA) — Criterio 1

Empezamos con un vistazo general: tipos de datos, estadísticas y datos faltantes.

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Verificamos si hay datos faltantes
df.isna().sum()

In [ ]:
# Revisamos el balance de clases de la variable objetivo
df['NObeyesdad'].value_counts()

El dataset no tiene datos faltantes, así que no es necesario eliminar filas ni imputar valores. Las clases están razonablemente balanceadas (entre ~270 y ~350 registros cada una).

**Variables numéricas:** `Age, Height, Weight, FCVC, NCP, CH2O, FAF, TUE`

**Variables categóricas:** `Gender, family_history_with_overweight, FAVC, CAEC, SMOKE, SCC, CALC, MTRANS`

### Visualizaciones

Relacionamos las variables con el target para identificar patrones, en vez de mirarlas de forma aislada.

In [ ]:
# Peso según nivel de obesidad
plt.figure(figsize=(10, 5))
sns.boxplot(x='NObeyesdad', y='Weight', data=df)
plt.xticks(rotation=45)
plt.title('Peso segun nivel de obesidad')
plt.tight_layout()
plt.show()

In [ ]:
# Historial familiar de sobrepeso vs nivel de obesidad
plt.figure(figsize=(8, 5))
sns.countplot(x='family_history_with_overweight', hue='NObeyesdad', data=df)
plt.title('Historial familiar de sobrepeso vs nivel de obesidad')
plt.show()

In [ ]:
# Relacion Altura vs Peso, coloreado por nivel de obesidad
plt.figure(figsize=(8, 6))
sns.scatterplot(x='Height', y='Weight', hue='NObeyesdad', data=df, alpha=0.7)
plt.title('Altura vs Peso por nivel de obesidad')
plt.show()

In [ ]:
# Correlacion entre variables numericas
plt.figure(figsize=(8, 6))
sns.heatmap(df[['Age','Height','Weight','FCVC','NCP','CH2O','FAF','TUE']].corr(),
            annot=True, cmap='coolwarm')
plt.title('Correlacion entre variables numericas')
plt.show()

*Agrega aquí tu interpretación de cada gráfica: ¿qué patrones observas? ¿qué variables parecen estar más relacionadas con el nivel de obesidad?*

---
## 4. Preprocesamiento de datos

El dataset tiene variables categóricas de distinto tipo, así que las tratamos de forma diferente.

### a) Variables binarias (yes/no) → 0/1

In [ ]:
binarias = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
for col in binarias:
    df[col] = df[col].map({'yes': 1, 'no': 0})

df[binarias].head()

### b) Variables ordinales (tienen un orden natural)

`CAEC` y `CALC` van de `no` a `Always`, así que las mapeamos a números en vez de usar codificación one-hot.

In [ ]:
orden = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
df['CAEC'] = df['CAEC'].map(orden)
df['CALC'] = df['CALC'].map(orden)

df[['CAEC', 'CALC']].head()

### c) Variables nominales (sin orden) → codificación one-hot

`Gender` y `MTRANS` no tienen un orden natural entre sus categorías.

In [ ]:
df = pd.get_dummies(df, columns=['Gender', 'MTRANS'], drop_first=True)
df.head()

### d) Codificar la variable objetivo

In [ ]:
le = LabelEncoder()
df['NObeyesdad_cod'] = le.fit_transform(df['NObeyesdad'])

# Vemos que numero corresponde a cada categoria
list(zip(le.classes_, range(len(le.classes_))))

### e) Separar X, y y dividir train/test

Usamos `stratify=y` para mantener las proporciones de las 7 clases en ambos conjuntos, igual que se hizo en el Caso 12.

In [ ]:
X = df.drop(columns=['NObeyesdad', 'NObeyesdad_cod'])
y = df['NObeyesdad_cod']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

### f) Escalado de variables

Necesario para KNN y SVM, que se basan en distancias entre observaciones.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

---
## 5. Selección e implementación de modelos — Criterio 2

La guía de la actividad menciona **SVM, Random Forest y KNN**. Entrenamos los tres y comparamos su desempeño.

**Justificación:** Random Forest suele funcionar bien en este tipo de problema porque no asume relaciones lineales entre las variables (por ejemplo, entre peso/altura y las 7 categorías de obesidad) y no requiere escalado. KNN y SVM sirven como puntos de comparación, ya que dependen de distancias entre observaciones.

In [ ]:
# Random Forest - no requiere datos escalados
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

In [ ]:
# KNN - requiere datos escalados
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
y_pred_knn = knn_model.predict(X_test_scaled)

In [ ]:
# SVM - requiere datos escalados
svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)

### Ajuste de hiperparámetros (opcional, nivel "Excelente" en la rúbrica)

Usamos `GridSearchCV` para buscar mejores parámetros de Random Forest, igual que se hizo con el número de vecinos de KNN en el Caso 11.

In [ ]:
param_grid = {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)
grid.fit(X_train, y_train)

print("Mejores parametros:", grid.best_params_)

In [ ]:
# Reentrenamos Random Forest con los mejores parametros encontrados
rf_model = RandomForestClassifier(**grid.best_params_, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

---
## 6. Evaluación y validación del modelo — Criterio 3

Con 7 clases (no 2 como en el caso de churn), nos apoyamos en `classification_report` (que ya incluye precisión, recall y F1 por cada clase) y la matriz de confusión, en vez de curva ROC.

In [ ]:
for nombre, y_pred in [('Random Forest', y_pred_rf), ('KNN', y_pred_knn), ('SVM', y_pred_svm)]:
    print(f'--- {nombre} ---')
    print('Accuracy:', accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    print()

In [ ]:
# Matriz de confusion del mejor modelo (ajusta segun tus resultados)
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Prediccion')
plt.ylabel('Real')
plt.title('Matriz de Confusion - Random Forest')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

*Agrega aquí tu interpretación: ¿cuál modelo tuvo mejor desempeño? ¿hay clases que se confunden entre sí (por ejemplo, niveles de obesidad consecutivos)?*

---
## 7. Interpretación de resultados y conclusiones — Criterio 4

### Importancia de variables

Nos muestra qué variables pesan más en las predicciones del modelo.

In [ ]:
importancias = pd.DataFrame({
    'Variable': X_train.columns,
    'Importancia': rf_model.feature_importances_
}).sort_values('Importancia', ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x='Importancia', y='Variable', data=importancias)
plt.title('Importancia de variables - Random Forest')
plt.tight_layout()
plt.show()

### Conclusiones

*Completa esta sección con tu propio análisis. Algunas preguntas guía:*

- ¿Cuál de los tres modelos (Random Forest, KNN, SVM) tuvo mejor accuracy y F1-score?
- ¿Qué variables resultaron más importantes para predecir el nivel de obesidad? ¿tiene sentido con lo que viste en el EDA?
- ¿El modelo confunde más algunas categorías que otras (por ejemplo, niveles de sobrepeso consecutivos)? ¿por qué crees que pasa?
- ¿Qué limitaciones tiene el modelo o los datos (por ejemplo, variables autoreportadas como hábitos alimenticios)?
- En términos prácticos, ¿para qué podría servir un modelo como este (ej. apoyo en tamizaje de salud pública)?

---
### Origen de los datos

- `obesity_dataset.csv` — Guía de actividades UA5, Universidad de La Sabana.